In [3]:
from math import lcm
from pprint import pp
from string import digits


def solution(signals):
    # 모든 신호등이 동시에 노란불(y)이 되는 가장 이른 시각(1초부터 시작)을 반환, 없으면 -1
    # 시각:  1 2 3 4 5 6 7 8 9 10 ...
    # 신호1: 1 y 4 4 4 4 1 y 4  4 ...   (g=1, y=1, r=4)
    # 신호2: 2 2 y 3 3 3 2 2 y  ...     (g=2, y=1, r=3)
    # 신호3: 3 3 3 y 2 2 3 3 3  y ...   (g=3, y=1, r=2)
    # 신호4: 4 4 4 4 y 1 4 4 4  4 ...   (g=4, y=1, r=1)
    cycles = [g + y + r for g, y, r in signals]
    total = lcm(*cycles)  # 전체 패턴이 반복되는 주기

    for t in range(1, total + 1):
        if all(g <= (t - 1) % c < g + y for (g, y, r), c in zip(signals, cycles)):
            return t
    return -1


s = [[1, 1, 4], [2, 1, 3], [3, 1, 2], [4, 1, 1]]
print(solution(s))  # -1

print(solution([[2, 1, 2], [5, 1, 1], [3, 3, 2]]))  # 8


-1
13


In [7]:
from collections import deque

def solution(prices):
        
    que = deque(prices)
    count = []
   
    while que:
        timer = 0
        a = que.popleft()
        for i in que:
            if a <= i:
                timer += 1
        count.append(timer)
    return count


p = [1, 2, 3, 2, 3]
solution(p)

[4, 3, 1, 1, 0]

In [5]:
def solution(prices):
    n = len(prices)
    answer = [0] * n
    stack = [] # (시점, 가격)

    for i, p in enumerate(prices):
        while stack and stack[-1][1] > p:
            j, _ = stack.pop()
            answer[j] = i - j
        stack.append((i,p))
    for j, _ in stack:
        answer[j] = n - 1 - j
    return answer
p = [1, 2, 3, 2, 3]
solution(p)

[4, 3, 1, 1, 0]

In [6]:
def solution(prices):
    answer = []
    for i in range(len(prices)):
        count = 0
        for j in range(i + 1, len(prices)):
            count += 1
            if prices[i] > prices[j]:
                break
        answer.append(count)
    return answer
p = [1, 2, 3, 2, 3]
solution(p)

[4, 3, 1, 1, 0]

In [4]:
from itertools import permutations, combinations, product

items = ["A", "B", "C"]

# 순열: 순서 O, 중복 X
print(list(permutations(items, 2)))

# 조합: 순서 X, 중복 X
print(list(combinations(items, 2)))

# 중복순열(데카르트 곱): 순서 O, 중복 O
print(list(product(items, repeat=2)))

[('A', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'C'), ('C', 'A'), ('C', 'B')]
[('A', 'B'), ('A', 'C'), ('B', 'C')]
[('A', 'A'), ('A', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'B'), ('B', 'C'), ('C', 'A'), ('C', 'B'), ('C', 'C')]


### 파이썬 완전탐색(Brute Force)과 DFS(깊이 우선 탐색)

완전탐색에서 모든 경우의 수를 만드는 방법은 크게 **`itertools` 라이브러리 활용**과 **DFS(재귀 + 백트래킹) 직접 구현** 두 가지로 나뉩니다.

#### 1. `itertools` vs DFS 언제 무엇을 쓸까?
| 구분 | `itertools` (`permutations`, `combinations`, `product`) | DFS / 백트래킹 (재귀 구현) |
| :--- | :--- | :--- |
| **장점** | 코드가 짧고 C로 구현되어 속도가 빠름 | 탐색 도중 조건이 안 맞으면 **가지치기(Pruning)** 가능 |
| **단점** | 모든 경우를 끝까지 다 만들어야만 확인 가능 | 재귀 깊이 제한(`sys.setrecursionlimit`) 및 구현 실수 주의 |
| **추천 상황** | $N$이 작고 단순 순열/조합 전체를 다 확인해도 될 때 | 탐색 중간에 불가능한 경로를 미리 차단해야 하거나 상태 변화가 복잡할 때 |

---

#### 2. DFS 백트래킹의 기본 뼈대
DFS로 경우의 수를 탐색할 때는 **상태 공간 트리(State Space Tree)**를 따라 내려가며 다음 3가지를 기억하면 됩니다.
1. **종료 조건 (Base Case)**: 원하는 개수($r$개)를 모두 골랐거나 끝에 도달했을 때 결과를 저장하고 `return`
2. **확장 및 방문 처리**: 아직 고르지 않은 후보를 선택(`path.append`, `visited[i] = True`)하고 다음 깊이로 재귀 호출(`dfs(depth + 1)`)
3. **상태 복구 (Backtracking)**: 재귀 호출이 끝나고 돌아오면 선택했던 것을 취소(`path.pop()`, `visited[i] = False`)하여 다른 경로에서도 사용할 수 있게 복원

In [5]:
items = ["A", "B", "C"]
r = 2

# 1. DFS로 순열(permutations) 구현: 순서 O, 중복 X -> visited 배열 사용
def dfs_permutations(items, r):
    result = []
    visited = [False] * len(items)
    path = []

    def dfs(depth):
        if depth == r:
            result.append(tuple(path))
            return
        for i in range(len(items)):
            if not visited[i]:
                visited[i] = True      # 방문(선택) 표시
                path.append(items[i])
                dfs(depth + 1)         # 다음 깊이로 탐색
                path.pop()             # 상태 복구 (백트래킹)
                visited[i] = False

    dfs(0)
    return result


# 2. DFS로 조합(combinations) 구현: 순서 X, 중복 X -> start 인덱스 사용
def dfs_combinations(items, r):
    result = []
    path = []

    def dfs(start, depth):
        if depth == r:
            result.append(tuple(path))
            return
        # start부터 탐색하여 이전에 고른 원소 이전으로는 돌아가지 않음 (순서 중복 방지)
        for i in range(start, len(items)):
            path.append(items[i])
            dfs(i + 1, depth + 1)      # 다음 원소는 i + 1부터 선택
            path.pop()

    dfs(0, 0)
    return result


# 3. DFS로 중복순열(product) 구현: 순서 O, 중복 O -> 제약 없이 매번 전체 탐색
def dfs_product(items, r):
    result = []
    path = []

    def dfs(depth):
        if depth == r:
            result.append(tuple(path))
            return
        for i in range(len(items)):
            path.append(items[i])
            dfs(depth + 1)             # 자기 자신(i)도 다시 선택 가능
            path.pop()

    dfs(0)
    return result


print("순열 (DFS):    ", dfs_permutations(items, r))
print("조합 (DFS):    ", dfs_combinations(items, r))
print("중복순열 (DFS):", dfs_product(items, r))

순열 (DFS):     [('A', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'C'), ('C', 'A'), ('C', 'B')]
조합 (DFS):     [('A', 'B'), ('A', 'C'), ('B', 'C')]
중복순열 (DFS): [('A', 'A'), ('A', 'B'), ('A', 'C'), ('B', 'A'), ('B', 'B'), ('B', 'C'), ('C', 'A'), ('C', 'B'), ('C', 'C')]


### DFS 완전탐색의 핵심 실전 패턴 2가지

코딩 테스트에서 `itertools`만으로 풀기 어렵고 **DFS를 직접 작성해야 하는 대표적인 유형**입니다.

#### 패턴 1: 원소별 선택 / 비선택 (부분집합 & 타겟 넘버 유형)
- 각 인덱스 `idx`의 원소를 **선택하거나(+), 선택하지 않거나(-)** 두 갈래로 분기합니다.
- 총 $2^N$가지 경우를 탐색하며, `path` 리스트를 따로 조작하지 않고 누적합(`current_sum`) 등을 인자로 바로 넘기면 백트래킹 코드가 훨씬 간결해집니다.
- **가지치기(Pruning)**: 탐색 도중 이미 목표치를 초과했거나 남은 원소를 다 더해도 목표에 도달할 수 없다면 즉시 `return`하여 탐색 속도를 크게 높일 수 있습니다.

#### 패턴 2: 그래프 / 2차원 격자(Grid) 탐색
- 인접 리스트나 2차원 배열에서 상하좌우(`dx`, `dy`)로 이동하며 연결된 영역의 크기나 경로의 수를 찾습니다.
- **단순 연결 요소 탐색**(단지 번호 붙이기, 섬의 개수 등)은 한 번 방문한 칸을 다시 방문하지 않으므로 `visited`를 해제하지 않습니다.
- **모든 경로 탐색**(출발지 $\rightarrow$ 도착지의 모든 경로, 최장 경로 등)은 다른 경로에서 해당 칸을 다시 밟을 수 있어야 하므로 재귀 호출 후 `visited`를 `False`로 되돌려야 합니다.

In [6]:
# [패턴 1] 원소별 선택/비선택 DFS + 가지치기(Pruning)
# 문제: 양수 리스트 nums에서 합이 정확히 target이 되는 부분집합들을 모두 구하기
def find_target_subsets(nums, target):
    result = []
    chosen = []

    def dfs(idx, current_sum):
        # 가지치기(Pruning): 모든 원소가 양수이므로 이미 target을 넘었다면 더 볼 필요 없음
        if current_sum > target:
            return

        # 종료 조건: 끝까지 다 확인했을 때
        if idx == len(nums):
            if current_sum == target:
                result.append(list(chosen))
            return

        # 1) 현재 원소(nums[idx])를 선택하는 경우
        chosen.append(nums[idx])
        dfs(idx + 1, current_sum + nums[idx])
        chosen.pop()  # 복구

        # 2) 현재 원소(nums[idx])를 선택하지 않는 경우
        dfs(idx + 1, current_sum)

    dfs(0, 0)
    return result


nums = [2, 3, 5, 7, 8]
print("합이 10이 되는 부분집합:", find_target_subsets(nums, 10))


# [패턴 2] 2차원 격자에서 모든 이동 경로의 수 구하기 (방문 해제 O)
# 0은 이동 가능, 1은 벽일 때 (0, 0)에서 (n-1, m-1)까지 가는 모든 단순 경로의 개수
def count_all_paths(grid):
    n, m = len(grid), len(grid[0])
    visited = [[False] * m for _ in range(n)]
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # 상, 하, 좌, 우
    count = 0

    def dfs(r, c):
        nonlocal count
        if r == n - 1 and c == m - 1:
            count += 1
            return

        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < m:
                if grid[nr][nc] == 0 and not visited[nr][nc]:
                    visited[nr][nc] = True
                    dfs(nr, nc)
                    visited[nr][nc] = False  # 다른 경로에서도 이 칸을 쓸 수 있도록 복구

    visited[0][0] = True
    dfs(0, 0)
    return count


sample_grid = [
    [0, 0, 0],
    [0, 1, 0],
    [0, 0, 0],
]
print("(0,0) -> (2,2) 가능한 경로 수:", count_all_paths(sample_grid))

합이 10이 되는 부분집합: [[2, 3, 5], [2, 8], [3, 7]]
(0,0) -> (2,2) 가능한 경로 수: 2


In [22]:
def solution(answ):
    answer = [0] * 3
    q = len(answ)
    a = [1, 2, 3, 4, 5]
    b = [2, 1, 2, 3, 2, 4, 2, 5]
    c = [3, 3, 1, 1, 2, 2, 4, 4, 5, 5]

    for i in range(q):
        u = answ[i]
        if u == a[i % len(a)]:
            answer[0] += 1
        if u == b[i % len(b)]:
            answer[1] += 1
        if u == c[i % len(c)]:
            answer[2] += 1

    result = []
    max_score = max(answer)
    for i, score in enumerate(answer, 1):
        if score == max_score:
            result.append(i)
    return result

a = [1,3,2,4,2]
solution(a)

[1, 2, 3]

In [18]:
from math import ceil

def solution(answers):
    scores = []
    persons = [[1, 2, 3, 4, 5], [2, 1, 2, 3, 2, 4, 2, 5], [3, 3, 1, 1, 2, 2, 4, 4, 5, 5]]

    for i, p in enumerate(persons):
        temp = ceil(len(answers) / len(p)) * p
        scores.append(sum(a == t for a, t in zip(answers, temp)))

        answer = []
        best = max(scores)
        for i, score in enumerate(scores):
            if score == best:
                answer.append(i + 1)
        return answer

a = [1, 3, 2, 4, 2]
solution(a)

[1]

In [28]:
def solution(n):
    # 0부터 n까지의 정수에 대해 소수 여부를 나타내는 배열 생성 (True로 초기화)
    # 0과 1은 소수가 아니므로 False로 설정
    is_prime = [True] * (n + 1)
    is_prime[0] = is_prime[1] = False

    # 2부터 n의 제곱근까지만 확인하면 충분합니다.
    for i in range(2, int(n**0.5) + 1):
        if is_prime[i]:  # i가 소수인 경우
            # i의 배수들을 모두 소수가 아닌 것(False)으로 제외
            for j in range(i * i, n + 1, i):
                is_prime[j] = False

    # True의 개수(소수의 개수)를 세어서 반환합니다.
    return is_prime.count(True)




n = 10
solution(n)

4

In [ ]:
from itertools import combinations
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

def solution(n):
    answer = 0

    for r in range(1, n + 1):

        if is_prime(r):
            answer += 1

    return answer

In [35]:
from itertools import permutations

def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

def solution(numbers):
    nums = set()
    for n in range(1, len(numbers) + 1):
        for temp in permutations(numbers, n):
            nums.add(int("".join(temp)))
    answer = 0

    for n in nums:
        if is_prime(n):
            answer += 1
    return answer

numbers = '17'
solution(numbers)

3